# ⏩ 使用 GitHub 模型的順序代理工作流程（Python）

## 📋 進階順序處理教學

本筆記本展示了使用 Microsoft Agent Framework 的<strong>順序工作流程模式</strong>。您將學習如何構建複雜的多步驟處理管線，其中代理依特定順序執行，並在階段間傳遞資料與上下文。

## 🎯 學習目標

### 🔄 <strong>順序處理模式</strong>
- <strong>線性工作流程設計</strong>：建立逐步處理管線
- <strong>資料流管理</strong>：在順序代理間傳遞資訊
- <strong>階段門處理</strong>：實作檢查點及驗證階段
- <strong>進度追蹤</strong>：監控工作流程執行及中間結果

### 🏗️ <strong>企業管線架構</strong>
- <strong>業務流程建模</strong>：將真實業務流程映射至代理工作流程
- <strong>品質保證</strong>：多階段驗證與審核流程
- <strong>文件處理</strong>：依序文件分析與轉換
- <strong>內容生產</strong>：具審核與核准階段的編輯工作流程

### 📊 <strong>進階工作流程功能</strong>
- <strong>上下文保存</strong>：跨工作流程階段維持狀態
- <strong>錯誤傳遞</strong>：處理順序處理中的失敗
- <strong>效能優化</strong>：高效的順序執行模式
- <strong>審計軌跡</strong>：完整追蹤順序操作

## ⚙️ 前置條件與設定

### 📦 <strong>依賴套件</strong>
```bash
pip install agent-framework-core -U
```

### 🔑 <strong>設定</strong>
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

## 🏢 <strong>企業順序工作流程使用案例</strong>

### 📝 <strong>文件處理管線</strong>
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 <strong>品質保證工作流程</strong>
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 <strong>內容生產管線</strong>
```
Research → Writing → Editing → Review → Publishing
```

### 💼 <strong>業務流程自動化</strong>
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 <strong>順序工作流程設計原則</strong>

- **🔗 線性推進**：每階段依賴前階段輸出
- **📋 狀態管理**：跨所有階段保存上下文與資料
- **🛡️ 錯誤處理**：在任一階段優雅處理失敗
- **📊 進度監控**：追蹤各階段完成度與效能
- **🔄 階段重用性**：設計可重複使用的工作流程元件

讓我們一起建構複雜的順序處理工作流程吧！🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    Message,
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = await provider.create_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = await provider.create_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = await provider.create_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal ChatMessage with an image of a
# living room. The current Message class no longer ships TextContent/DataContent
# helpers, so this migration uses a textual description of the same scene to
# keep the lesson focused on sequential workflow mechanics.
message = Message(
    role="user",
    text=(
        "I am furnishing a modern living room and want pieces that fit a warm, "
        "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
        "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
        "Please find appropriate furniture and give the corresponding price for "
        "each piece, then produce a final purchase quote."
    ),
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
